# 01 · Ingesta a bronze

**Propósito:** copiar las tablas de `samples.tpch` a la capa bronze, agregando columnas de linaje.

**Entradas:** `samples.tpch.{orders, lineitem, part, partsupp, supplier, nation, region}`

**Salidas:** `bronze_<tabla>` (overwrite), filas en `bronze_ingestion_log`, task value `bronze_row_counts`

**Recorte por fecha:** `orders` se filtra por `o_orderdate` entre `date_from` y `date_to` de la config; `lineitem` se reduce con un semi-join contra esos pedidos. Las dimensiones se copian completas.

In [ ]:
%run ./00_config

In [ ]:
import json
import os

from pyspark.sql import functions as F

from src.utils.spark_helpers import run_sql_file, with_ingestion_metadata, write_delta

SOURCE_SCHEMA = "samples.tpch"

run_sql_file(
    spark,
    os.path.join(SQL_DIR, "ddl_bronze.sql"),
    {
        "ingestion_log_table": cfg.table("bronze_ingestion_log"),
        "dq_results_table": cfg.table("dq_results"),
    },
)

## Hechos: pedidos y líneas de pedido

In [ ]:
orders = spark.table(f"{SOURCE_SCHEMA}.orders").filter(
    F.col("o_orderdate").between(F.lit(cfg.date_from), F.lit(cfg.date_to))
)
lineitem = spark.table(f"{SOURCE_SCHEMA}.lineitem").join(
    orders.select(F.col("o_orderkey").alias("l_orderkey")),
    on="l_orderkey",
    how="left_semi",
)

sources = {
    "orders": orders,
    "lineitem": lineitem,
}

## Dimensiones

In [ ]:
for name in ("part", "partsupp", "supplier", "nation", "region"):
    sources[name] = spark.table(f"{SOURCE_SCHEMA}.{name}")

## Escritura y bitácora

In [ ]:
row_counts = {}
for name, df in sources.items():
    target = cfg.table(f"bronze_{name}")
    write_delta(with_ingestion_metadata(df, f"{SOURCE_SCHEMA}.{name}"), target)
    row_counts[name] = spark.table(target).count()
    log.info("Loaded %s -> %s (%s rows)", name, target, row_counts[name])

log_rows = [
    (RUN_ID, f"{SOURCE_SCHEMA}.{name}", cfg.table(f"bronze_{name}"), count)
    for name, count in row_counts.items()
]
(
    spark.createDataFrame(log_rows, "run_id string, source_table string, target_table string, row_count bigint")
    .withColumn("ingested_at", F.current_timestamp())
    .write.mode("append")
    .saveAsTable(cfg.table("bronze_ingestion_log"))
)

In [ ]:
dbutils.jobs.taskValues.set(key="bronze_row_counts", value=row_counts)
dbutils.notebook.exit(json.dumps({"bronze_row_counts": row_counts}))